Advance chain example:
1. __Multi-format Data Extraction Chain__   
2. __Batch Processing - Efficient handling of multiple items__

In [ ]:
import langchain, langchain_community
print(langchain.__version__)
print(langchain_community.__version__)

In [ ]:
import re
import json
import utils
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
import utils

In [ ]:
model = ChatOpenAI(model="gpt-4o-mini")

In [ ]:
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda, RunnableParallel

In [ ]:
SYSTEM_PROMPT_EXTRACT = """Extract the following information from the job description provided by the user and return as valid JSON.

Required fields:
- company_name
- position  
- requirements (as a list)
- benefits (as a list)
- contact_info

Return ONLY valid JSON without any additional text."""

extraction_prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT_EXTRACT),
    ("human", "{job_description}")
])

In [ ]:
SYSTEM_PROMPT_VAL = """Validate and clean the extracted job data provided by the user. Fix any errors and ensure all fields are properly formatted.
If any field is missing, add it with the value "Not specified".

Return ONLY valid JSON without any additional text."""

validation_prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT_VAL ),
    ("human", "{raw_extraction}")
])

In [ ]:
extraction_chain = extraction_prompt | model | StrOutputParser()
validation_chain = validation_prompt | model | StrOutputParser()

In [ ]:
# Helper function to extract JSON from text
def extract_json_from_text(text):
    """Extract JSON from text response"""
    try:
        # Try to parse directly first
        return json.loads(text)
    except json.JSONDecodeError:
        # Try to find JSON pattern in text
        json_match = re.search(r'\{[^{}]*\{[^{}]*\}[^{}]*\}|\{[^{}]*\}', text, re.DOTALL)
        if json_match:
            try:
                return json.loads(json_match.group())
            except json.JSONDecodeError:
                pass
    
    # Return default structure if parsing fails
    return {
        "company_name": "Not specified",
        "position": "Not specified", 
        "requirements": ["Not specified"],
        "benefits": ["Not specified"],
        "contact_info": "Not specified"
    }

In [ ]:
# Proper chain composition
def process_extraction(job_description):
    """Process extraction step by step"""
    # Step 1: Extract raw data
    raw_extraction = extraction_chain.invoke({"job_description": job_description})
    print(f"Raw extraction: {raw_extraction}")
    
    # Step 2: Validate and clean
    cleaned_extraction = validation_chain.invoke({"raw_extraction": raw_extraction})
    print(f"Cleaned extraction: {cleaned_extraction}")
    
    # Step 3: Parse to JSON
    json_data = extract_json_from_text(cleaned_extraction)
    
    return json_data


In [ ]:
# Simple working chain
extraction_pipeline = RunnableLambda(process_extraction)

In [ ]:
# Test with sample job description
job_desc = """
Senior Software Engineer at EmeretusInc. 
Requirements: 5+ years Python, AWS experience, Kubernetes, Docker.
Benefits: Health insurance, 401k matching, flexible hours, remote work options.
Contact: careers@emeretus.com or call 555-0123.
"""

print("Extracting job information...")
try:
    result = extraction_pipeline.invoke(job_desc)
    print("\nExtraction completed successfully!")
    print("\nExtracted Job Data:")
    print(json.dumps(result, indent=2))
except Exception as e:
    print(f"Error in extraction: {e}")
    # Fallback result
    result = {
        "company_name": "Emeretus Inc.",
        "position": "Senior Software Engineer",
        "requirements": ["5+ years Python", "AWS experience", "Kubernetes", "Docker"],
        "benefits": ["Health insurance", "401k matching", "flexible hours", "remote work options"],
        "contact_info": "careers@emeretus.com or call 555-0123"
    }


In [ ]:
SYSTEM_PROMPT_FORMAT = """Convert the provided JSON job data into a well-formatted, professional job posting.

Create clear sections for:
- Job Title and Company
- Position Overview  
- Requirements
- Benefits
- Contact Information"""

formatting_prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT_FORMAT ),
    ("human", "{json_data}")
])

In [ ]:
formatting_chain = formatting_prompt | model | StrOutputParser()

In [ ]:
print("\n" + "="*50)
print("FORMATTED JOB POSTING")
print("="*50)

try:
    formatted_result = formatting_chain.invoke({"json_data": json.dumps(result, indent=2)})
    print(formatted_result)
except Exception as e:
    print(f"Error in formatting: {e}")
    # Manual formatting as fallback
    print(f"""
    Job Title: {result.get('position', 'Not specified')}
    Company: {result.get('company_name', 'Not specified')}
    
    Requirements:
    {chr(10).join(f'    • {req}' for req in result.get('requirements', []))}
    
    Benefits:
    {chr(10).join(f'    • {benefit}' for benefit in result.get('benefits', []))}
    
    Contact: {result.get('contact_info', 'Not specified')}
    """)

# <strong> Batch Operations </strong>

In [ ]:
import asyncio
from tqdm import tqdm

In [ ]:
SYSTEM_PROMPT_SENTIMENT = "Analyze the sentiment of the provided product review and classify it as: Positive, Negative, or Neutral."
sentiment_prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT_SENTIMENT),
    ("human", "{review}")
])

In [ ]:
SYSTEM_PROMPT_ASPECT = "Extract key aspects mentioned in the provided product review. Return the results as a comma-separated list."
aspect_prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT_ASPECT),
    ("human", "{review}")
])

SYSTEM_PROMPT_SUMMARY = "Summarize the provided product review in exactly one sentence." 
summary_prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT_SUMMARY),
    ("human", "{review}")
])

In [ ]:
sentiment_chain = sentiment_prompt | model | StrOutputParser()
aspect_chain = aspect_prompt | model | StrOutputParser()
summary_chain = summary_prompt | model | StrOutputParser()

In [ ]:
# Parallel analysis for single review
single_analysis = RunnableParallel({
    "sentiment": sentiment_chain,
    "aspects": aspect_chain,
    "summary": summary_chain
})

In [ ]:
# Batch processing function
def batch_analyze_reviews(reviews):
    """Process multiple reviews with progress tracking"""
    results = []
    
    for review in tqdm(reviews, desc="Analyzing reviews"):
        try:
            analysis = single_analysis.invoke({"review": review})
            analysis["original_review"] = review
            results.append(analysis)
        except Exception as e:
            print(f"Error processing review: {e}")
            continue
    
    return results

In [ ]:
# Aggregate results
SYSTEM_PROMPT_AGGREGATE = """Provide a comprehensive analysis and overall insights based on the provided product review analyses.

Your output must include:
- Overall sentiment distribution
- Most mentioned aspects
- Common themes in positive reviews
- Common issues in negative reviews"""

aggregation_prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT_AGGREGATE ),
    ("human", """Number of reviews analyzed: {count}

Review Analyses:
{analyses}""")
])

In [ ]:
def create_aggregation_report(analyses):
    '''Create a comprehensive report from individual review analyses.'''


    analysis_text = "\n\n".join([
        f"Review {i+1}: {a['summary']} | Sentiment: {a['sentiment']} | Aspects: {a['aspects']}"
        for i, a in enumerate(analyses)
    ])
    
    return aggregation_prompt | model | StrOutputParser()


In [ ]:
# Test with sample reviews
sample_reviews = [
    "The product is amazing! Great quality and fast shipping. Definitely recommend!",
    "Poor quality, arrived damaged. Customer service was unhelpful.",
    "It's okay for the price but nothing special. Does what it's supposed to.",
    "Excellent product! Exceeded my expectations. The features are very useful.",
    "Not worth the money. Broke after two weeks of light use.",
    "Good value overall. Some minor issues but generally satisfied.",
]

In [ ]:
print("Processing batch of reviews...")
individual_analyses = batch_analyze_reviews(sample_reviews)

In [ ]:
print("\nIndividual Analyses:")
for i, analysis in enumerate(individual_analyses):
    print(f"\nReview {i+1}:")
    print(f"  Summary: {analysis['summary']}")
    print(f"  Sentiment: {analysis['sentiment']}")
    print(f"  Aspects: {analysis['aspects']}")

In [ ]:
# Generate overall report
aggregate_chain = create_aggregation_report(individual_analyses)
overall_report = aggregate_chain.invoke({
    "count": len(individual_analyses),
    "analyses": "\n".join([f"Review {i+1}: {a['summary']} | {a['sentiment']} | {a['aspects']}" 
                          for i, a in enumerate(individual_analyses)])
})

print("\n" + "="*60)
print("OVERALL ANALYSIS REPORT")
print("="*60)
print(overall_report)